In [55]:
# ============================================================
# 036_daily_orchestrator
# ============================================================
#
# Overview
# ----------------
# Executes the full daily researchOS pipeline in a single orchestrated run.
# Runs component notebooks sequentially (config, scanner, PDF processor,
# monitors), captures execution status and metrics, handles partial failures,
# and produces a comprehensive Daily Run Summary in three formats:
# Notion-ready Markdown, standard Markdown, and Slack text snippet.
#
# Inputs / Outputs
# ----------------
# Inputs:
#   - Component notebooks: 028_config_and_state, 030_daily_paper_scanner,
#     031_pdf_inbox_processor, 032_monitor_vc_daily, 033_monitor_startups_daily,
#     034_monitor_policy_daily, 035_monitor_people_daily
#   - .env file (consumed by 028)
#
# Outputs:
#   - Orchestration execution log (per-module status, metrics, errors)
#   - daily_summary_md_notion: Markdown optimized for Notion blocks
#   - daily_summary_md: Standard Markdown summary
#   - daily_summary_slack: Slack-formatted text snippet
#   - orchestrator_state dict: aggregated metrics and status
#
# Structure
# ----------------
# Cell 01: Import dependencies and define execution strategy
# Cell 02: Define module registry and execution configuration
# Cell 03: Initialize orchestrator state and execution context
# Cell 04: Execute 028_config_and_state (required foundation)
# Cell 05: Execute 030_daily_paper_scanner with error handling
# Cell 06: Execute 031_pdf_inbox_processor with error handling
# Cell 07: Execute 032_monitor_vc_daily with error handling
# Cell 08: Execute 033_monitor_startups_daily with error handling
# Cell 09: Execute 034_monitor_policy_daily with error handling
# Cell 10: Execute 035_monitor_people_daily with error handling
# Cell 11: Aggregate metrics from all module executions
# Cell 12: Generate Notion-ready Markdown summary
# Cell 13: Generate standard Markdown summary
# Cell 14: Generate Slack text snippet summary
# Cell 15: Display final summary and save artifacts
#
# Notes
# ----------------
# - Execution model:
#     - 028_config_and_state is executed first to initialize run_id, logger, and state helpers.
#     - Component notebooks may execute in a separate kernel/process via execute_notebook().
#       Therefore, orchestrator SHOULD NOT rely on in-kernel globals being populated by modules.
#
# - Metrics strategy (important):
#     - Prefer artifact-based metrics (e.g., artifacts/summaries/{run_id}_summary.json)
#       to make aggregation robust across separate kernels.
#     - Diff-based globals metrics (snapshot_keys/diff_metrics) are treated as a fallback only.
#
# - Failure handling:
#     - Required modules (e.g., 028 and 029 if enabled) halt the pipeline on failure.
#     - Non-critical modules (030–035) record errors and continue when continue_on_failure=True.
#
# - Idempotency:
#     - Downstream notebooks are rerun-safe via dedup_key logic (in-memory + Notion dedup).
#
# - Defensive coding:
#     - Uses .get() and setdefault to avoid KeyError on missing exports.
#     - Module metrics are normalized into orchestrator_state['module_results'][name]['metrics'].
#
# - Side effects:
#     - Modules 032–035 accumulate Events into Notion via 029 wrappers only (no raw Notion calls here).
#     - Summary artifacts are saved for Notion/Markdown/Slack plus orchestrator_state JSON.


In [38]:
# ============================================================
# Cell 01 — Import dependencies and define execution strategy
# ============================================================
# Overview:
#   Import standard libraries, define orchestration helpers,
#   and establish the execution strategy (sequential %run with
#   error handling and metrics capture).
#
# Inputs / Outputs:
#   Outputs: execute_notebook(), capture_module_metrics()
#
# Notes:
#   - Uses %run magic for in-kernel notebook execution
#   - Each module runs in shared namespace to preserve 028 state
#

# --- Mandatory env loading ---
from dotenv import load_dotenv
load_dotenv('env.txt')

# --- Runtime LLM configuration (given / assumed) ---
llm_provider = 'OpenAI'
llm_model = 'gpt-4o-mini'
llm_temperature = 0.0

import os
import sys
from datetime import datetime, timezone
from typing import Dict, Any, List, Optional
import traceback
import time

# Execution strategy: sequential %run with shared kernel state
# to preserve globals from 028_config_and_state.

def execute_notebook(notebook_path: str, module_name: str) -> Dict[str, Any]:
    """
    Execute a notebook using %run magic and capture execution metadata.
    
    Returns dict with:
      - status: 'success' | 'failed' | 'skipped'
      - start_time: ISO timestamp
      - end_time: ISO timestamp
      - duration_seconds: float
      - error: Optional[str]
    """
    result = {
        'module': module_name,
        'notebook': notebook_path,
        'status': 'skipped',
        'start_time': None,
        'end_time': None,
        'duration_seconds': 0.0,
        'error': None
    }
    
    if not os.path.exists(notebook_path):
        result['status'] = 'skipped'
        result['error'] = f'Notebook not found: {notebook_path}'
        return result
    
    start = time.time()
    result['start_time'] = datetime.now(timezone.utc).isoformat()
    
    try:
        # Execute notebook in shared kernel namespace
        get_ipython().run_line_magic('run', notebook_path)
        result['status'] = 'success'
    except Exception as e:
        result['status'] = 'failed'
        result['error'] = f'{type(e).__name__}: {str(e)}'
        # Log traceback for debugging
        print(f"\n[ERROR] {module_name} failed:")
        traceback.print_exc()
    
    end = time.time()
    result['end_time'] = datetime.now(timezone.utc).isoformat()
    result['duration_seconds'] = round(end - start, 2)
    
    return result

def snapshot_keys():
    # 追跡したいキーだけに絞る（ノイズ削減）
    return {
        "papers_found": globals().get("papers_found"),
        "pdfs_processed": globals().get("pdfs_processed"),
        "updates_found": globals().get("updates_found"),
        "events_created": globals().get("events_created"),
        "created_events": globals().get("created_events"),
        "run_stats": globals().get("run_stats"),
        "scan_stats": globals().get("scan_stats"),
    }

def diff_metrics(before: dict, after: dict) -> dict:
    out = {}
    for k, v_after in after.items():
        v_before = before.get(k, None)
        # 値が変わった or beforeになかったものだけ採用
        if k not in before or v_after != v_before:
            # dict なら shallow copy（参照事故防止）
            if isinstance(v_after, dict):
                out[k] = dict(v_after)
            else:
                out[k] = v_after
    return out

def capture_module_metrics(module_name: str, globals_dict: dict) -> Dict[str, Any]:
    """
    Extract exported metrics from a module's global variables.
    
    Looks for common export patterns:
      - {module}_results, {module}_metrics, {module}_summary
      - Counts: papers_found, events_created, etc.
    
    Returns dict of captured metrics (empty if none found).
    """
    metrics = {}
    
    # Common export variable patterns
    candidates = [
        f'{module_name}_results',
        f'{module_name}_metrics',
        f'{module_name}_summary',
        'papers_found',
        'pdfs_processed',
        'events_created',
        'updates_found',
        'items_processed'
    ]
    
    for key in candidates:
        if key in globals_dict:
            val = globals_dict[key]
            # Store int/str/dict, skip complex objects
            if isinstance(val, (int, str, dict, list)):
                metrics[key] = val
    
    return metrics

def get_module(name: str) -> dict:
    m = next((x for x in MODULE_REGISTRY if x.get("name") == name), None)
    if m is None:
        raise RuntimeError(f"MODULE_REGISTRY missing module name='{name}'")
    return m


print('[Cell 01] Orchestration helpers loaded.')
print(f'  - execute_notebook(): %run wrapper with error handling')
print(f'  - capture_module_metrics(): extract exported variables')


[Cell 01] Orchestration helpers loaded.
  - execute_notebook(): %run wrapper with error handling
  - capture_module_metrics(): extract exported variables


In [32]:
# ============================================================
# Cell 02 — Define module registry and execution configuration
# ============================================================
# Overview:
#   Define the ordered list of component modules to execute,
#   their notebook paths, and execution parameters.
#   Establishes the dependency chain and retry policy.
#
# Inputs / Outputs:
#   Outputs: MODULE_REGISTRY (list of dicts), EXECUTION_CONFIG (dict)
#
# Notes:
#   - 028 is marked required=True; pipeline halts if it fails
#   - Other modules continue on failure to maximize data collection
#   - Execution order matters: 028 first, then scanners/processors/monitors
#

# Module registry: ordered list of notebooks to execute
MODULE_REGISTRY = [
    {"name": "config_and_state", "notebook": "028_config_and_state.ipynb", "required": True},
    {"name": "notion_clients_and_io", "notebook": "029_notion_clients_and_io.ipynb", "required": True},
    {"name": "daily_paper_scanner", "notebook": "030_daily_paper_scanner.ipynb", "required": False},
    {"name": "pdf_inbox_processor", "notebook": "031_pdf_inbox_processor.ipynb", "required": False},
    {"name": "monitor_vc_daily", "notebook": "032_monitor_vc_daily.ipynb", "required": False},
    {"name": "monitor_startups_daily", "notebook": "033_monitor_startup_daily.ipynb", "required": False},
    {"name": "monitor_policy_daily", "notebook": "034_monitor_policy_daily.ipynb", "required": False},
    {"name": "monitor_people_daily", "notebook": "035_monitor_people_daily.ipynb", "required": False},
]


# Execution configuration
EXECUTION_CONFIG = {
    'continue_on_failure': True,  # Continue pipeline even if non-required modules fail
    'max_retries': 0,             # No automatic retries (idempotency handled by modules)
    'timeout_seconds': None,      # No timeout (trust module execution time)
    'log_level': 'INFO',          # Logging verbosity
    'capture_metrics': True,      # Extract metrics from module globals
    'notebooks_dir': '.'          # Base directory for notebook paths
}

print('[Cell 02] Module registry and execution config defined.')
print(f'  - Total modules: {len(MODULE_REGISTRY)}')
print(f'  - Required modules: {sum(1 for m in MODULE_REGISTRY if m["required"])}')
print(f'  - Continue on failure: {EXECUTION_CONFIG["continue_on_failure"]}')
print(f'\nExecution order:')
for i, module in enumerate(MODULE_REGISTRY, 1):
    req = ' [REQUIRED]' if module['required'] else ''
    print(f'  {i}. {module["name"]}{req}')
    print(f'     → {module["notebook"]}')


[Cell 02] Module registry and execution config defined.
  - Total modules: 8
  - Required modules: 2
  - Continue on failure: True

Execution order:
  1. config_and_state [REQUIRED]
     → 028_config_and_state.ipynb
  2. notion_clients_and_io [REQUIRED]
     → 029_notion_clients_and_io.ipynb
  3. daily_paper_scanner
     → 030_daily_paper_scanner.ipynb
  4. pdf_inbox_processor
     → 031_pdf_inbox_processor.ipynb
  5. monitor_vc_daily
     → 032_monitor_vc_daily.ipynb
  6. monitor_startups_daily
     → 033_monitor_startup_daily.ipynb
  7. monitor_policy_daily
     → 034_monitor_policy_daily.ipynb
  8. monitor_people_daily
     → 035_monitor_people_daily.ipynb


In [23]:
# ============================================================
# Cell 03 — Initialize orchestrator state and execution context
# ============================================================
# Overview:
#   Initialize the orchestrator_state dictionary to track execution
#   across all modules. Records pipeline start time, module results,
#   aggregated metrics, and summary output formats.
#
# Inputs / Outputs:
#   Outputs: orchestrator_state (dict), pipeline_start_time (datetime)
#
# Notes:
#   - orchestrator_state persists across all module executions
#   - Stores per-module execution results from execute_notebook()
#   - Accumulates metrics for final summary generation
#   - Structure matches summary generation requirements in cells 12-14
#

# Initialize pipeline start time
pipeline_start_time = datetime.now(timezone.utc)

# Initialize orchestrator state dictionary
orchestrator_state = {
    # Pipeline-level metadata
    'pipeline_start': pipeline_start_time.isoformat(),
    'pipeline_end': None,
    'pipeline_duration_seconds': 0.0,
    'pipeline_status': 'running',  # 'running' | 'completed' | 'failed'
    
    # Per-module execution results
    # Each entry populated by execute_notebook() in cells 04-10
    'module_results': {},
    
    # Aggregated metrics across all modules
    # Populated in cell 11 from module_results and captured globals
    'aggregated_metrics': {
        'total_modules_executed': 0,
        'modules_succeeded': 0,
        'modules_failed': 0,
        'modules_skipped': 0,
        'total_execution_time': 0.0,
        # Module-specific counts (populated from globals)
        'papers_found': 0,
        'pdfs_processed': 0,
        'events_created': 0,
        'vc_updates': 0,
        'startup_updates': 0,
        'policy_updates': 0,
        'people_updates': 0
    },
    
    # Summary outputs (generated in cells 12-14)
    'summaries': {
        'notion_markdown': None,
        'standard_markdown': None,
        'slack_text': None
    },
    
    # Error tracking
    'errors': [],  # List of error dicts for failed modules
    
    # Run identification (will be populated from 028 globals after execution)
    'run_id': None,
    'config_loaded': False
}

print('[Cell 03] Orchestrator state initialized.')
print(f'  - Pipeline start: {orchestrator_state["pipeline_start"]}')
print(f'  - Modules to execute: {len(MODULE_REGISTRY)}')
print(f'  - State structure ready for metric aggregation')
print(f'\nInitial state keys: {list(orchestrator_state.keys())}')


[Cell 03] Orchestrator state initialized.
  - Pipeline start: 2026-02-04T06:54:59.205777+00:00
  - Modules to execute: 8
  - State structure ready for metric aggregation

Initial state keys: ['pipeline_start', 'pipeline_end', 'pipeline_duration_seconds', 'pipeline_status', 'module_results', 'aggregated_metrics', 'summaries', 'errors', 'run_id', 'config_loaded']


In [24]:
# ============================================================
# Cell 04 — Execute 028_config_and_state (required foundation)
# ============================================================
# Overview:
#   Execute 028_config_and_state.ipynb to initialize global configuration,
#   state management, and logging infrastructure. This is a required module;
#   if it fails, the entire pipeline halts.
#
# Inputs / Outputs:
#   Inputs: MODULE_REGISTRY[0], orchestrator_state
#   Outputs: Updated orchestrator_state['module_results']['config_and_state'],
#            orchestrator_state['run_id'], orchestrator_state['config_loaded']
#
# Notes:
#   - 028 exports critical globals: run_id, logger, state_manager, config
#   - Pipeline cannot continue if 028 fails (required=True)
#   - After execution, capture run_id and config_loaded status
#   - Uses execute_notebook() and capture_module_metrics() from Cell 01
#

print('[Cell 04] Executing 028_config_and_state (REQUIRED)...')
print('=' * 60)

# Get module configuration from registry
config_module = MODULE_REGISTRY[0]
assert config_module['name'] == 'config_and_state', 'First module must be config_and_state'
assert config_module['required'], 'config_and_state must be marked required'

# Execute the notebook
notebook_path = os.path.join(EXECUTION_CONFIG['notebooks_dir'], config_module['notebook'])
result = execute_notebook(notebook_path, config_module['name'])

# Store execution result
orchestrator_state['module_results'][config_module['name']] = result

# Check execution status
if result['status'] == 'failed':
    error_msg = f"CRITICAL: {config_module['name']} failed (required module)"
    print(f"\n{'=' * 60}")
    print(f"[ERROR] {error_msg}")
    print(f"Error details: {result.get('error', 'Unknown error')}")
    print(f"{'=' * 60}")
    
    # Record error and halt pipeline
    orchestrator_state['errors'].append({
        'module': config_module['name'],
        'error': result.get('error'),
        'timestamp': result.get('end_time')
    })
    orchestrator_state['pipeline_status'] = 'failed'
    orchestrator_state['pipeline_end'] = datetime.now(timezone.utc).isoformat()
    
    raise RuntimeError(f"{error_msg}: {result.get('error')}")

elif result['status'] == 'skipped':
    error_msg = f"CRITICAL: {config_module['name']} skipped (required module)"
    print(f"\n{'=' * 60}")
    print(f"[ERROR] {error_msg}")
    print(f"Reason: {result.get('error', 'Notebook not found')}")
    print(f"{'=' * 60}")
    
    orchestrator_state['pipeline_status'] = 'failed'
    orchestrator_state['pipeline_end'] = datetime.now(timezone.utc).isoformat()
    
    raise RuntimeError(error_msg)

else:
    # Success: capture exported globals from 028
    print(f"\n[SUCCESS] {config_module['name']} completed in {result['duration_seconds']}s")
    
    # Capture metrics from module execution
    if EXECUTION_CONFIG['capture_metrics']:
        metrics = capture_module_metrics(config_module['name'], globals())
        result['metrics'] = metrics
        print(f"  - Captured metrics: {list(metrics.keys()) if metrics else 'none'}")
    
    # Extract critical globals exported by 028
    # These are used by downstream modules and summary generation
    if 'run_id' in globals():
        orchestrator_state['run_id'] = globals()['run_id']
        print(f"  - Run ID: {orchestrator_state['run_id']}")
    else:
        print("  - WARNING: run_id not found in globals (expected from 028)")
    
    # Mark config as loaded
    orchestrator_state['config_loaded'] = True
    
    # Check for expected global objects
    expected_globals = ['logger', 'state_manager', 'config', 'run_id']
    found_globals = [g for g in expected_globals if g in globals()]
    missing_globals = [g for g in expected_globals if g not in globals()]
    
    print(f"  - Found globals: {found_globals}")
    if missing_globals:
        print(f"  - WARNING: Missing expected globals: {missing_globals}")

print('\n' + '=' * 60)
print('[Cell 04] 028_config_and_state execution completed.')
print(f"  - Status: {result['status']}")
print(f"  - Duration: {result['duration_seconds']}s")
print(f"  - Config loaded: {orchestrator_state['config_loaded']}")
print(f"  - Run ID: {orchestrator_state.get('run_id', 'N/A')}")
print('=' * 60)


[Cell 04] Executing 028_config_and_state (REQUIRED)...
✓ Cell 01: Imports and dependencies loaded
✓ Cell 02: Environment bootstrap completed
  - NOTION_TOKEN: ntn_38...***
  - NOTION_VERSION: 2025-09-03
  - NOTION_LIT_DB_ID: set
  - NOTION_EVENTS_DB_ID: set
  - NOTION_MONITORING_TARGETS_DB_ID: set
  - NOTION_MONITORING_QUEUE_DB_ID: set
✓ Cell 03: Loaded configuration from config.yaml
  - Drive folder_id:      (set)
✓ Cell 03: Configuration validated (source: config.yaml)
  - Pipeline cadence:     daily
  - Max runtime:          30 min
  - Lookback (daily):     7 days
  - Lookback (tasks):     14 days
  - Lookback (projects):  30 days
  - Max items per query:  100
  - Logging level:        INFO
✓ Cell 04: Run context initialized
  - Run ID:           81d104d1-2541-4bea-8eb7-262e808fa381
  - Execution start:  2026-02-04T06:55:00.329917+00:00
  - Run date:         2026-02-04
  - Timezone:         UTC
  - Cadence:          daily
  - Lookback windows:
    - daily_notes : 7 days (2026-01-28 

In [25]:
# ============================================================
# Cell 04.5 — Execute 029_notion_clients_and_io (required foundation)
# ============================================================
# Overview:
#   Execute 029_notion_clients_and_io.ipynb to initialize Notion REST client utilities,
#   retry/backoff, schema introspection, and CRUD wrappers used by downstream notebooks.
#   This is required for stable pipeline execution.
#
# Inputs / Outputs:
#   Inputs: MODULE_REGISTRY[1], orchestrator_state, globals() (expects 028 already ran)
#   Outputs: orchestrator_state['module_results']['notion_clients_and_io'],
#            notion wrapper functions bound into current globals (if shared-kernel)
#
# Notes:
#   - 029 owns ALL Notion I/O. Orchestrator must not re-implement Notion calls.
#   - If 029 fails, downstream notebooks that rely on wrappers will likely fail.
#   - Whether globals are shared depends on execute_notebook() implementation.
#

print('[Cell 04.5] Executing 029_notion_clients_and_io (REQUIRED)...')
print('=' * 60)

notion_module = MODULE_REGISTRY[1]
assert notion_module['name'] == 'notion_clients_and_io', 'Second module must be notion_clients_and_io'
assert notion_module['required'], 'notion_clients_and_io must be marked required'

notebook_path = os.path.join(EXECUTION_CONFIG['notebooks_dir'], notion_module['notebook'])
result = execute_notebook(notebook_path, notion_module['name'])

orchestrator_state['module_results'][notion_module['name']] = result

if result['status'] in ('failed', 'skipped'):
    error_msg = f"CRITICAL: {notion_module['name']} {result['status']} (required module)"
    print(f"\n{'=' * 60}")
    print(f"[ERROR] {error_msg}")
    print(f"Details: {result.get('error', 'Unknown error')}")
    print(f"{'=' * 60}")

    orchestrator_state['errors'].append({
        'module': notion_module['name'],
        'error': result.get('error'),
        'timestamp': result.get('end_time')
    })
    orchestrator_state['pipeline_status'] = 'failed'
    orchestrator_state['pipeline_end'] = datetime.now(timezone.utc).isoformat()

    raise RuntimeError(f"{error_msg}: {result.get('error')}")

print(f"\n[SUCCESS] {notion_module['name']} completed in {result['duration_seconds']}s")

# Optional: capture metrics / wrapper presence
if EXECUTION_CONFIG.get('capture_metrics'):
    metrics = capture_module_metrics(notion_module['name'], globals())
    result['metrics'] = metrics
    print(f"  - Captured metrics: {list(metrics.keys()) if metrics else 'none'}")

# Optional: sanity check for commonly used wrappers
expected_wrappers = [
    'create_event', 'query_monitoring_targets', 'create_paper_inbox',
    'query_event_by_dedup_key', 'update_paper_links'
]
found = [w for w in expected_wrappers if w in globals() and callable(globals()[w])]
missing = [w for w in expected_wrappers if w not in found]
print(f"  - Wrapper check (callable): found={found}")
if missing:
    print(f"  - NOTE: Some wrappers not found (may be OK if naming differs): {missing}")

print('\n' + '=' * 60)
print('[Cell 04.5] 029_notion_clients_and_io execution completed.')
print(f"  - Status: {result['status']}")
print(f"  - Duration: {result['duration_seconds']}s")
print('=' * 60)


[Cell 04.5] Executing 029_notion_clients_and_io (REQUIRED)...
2026-02-04 15:55:05 | INFO     | 81d104d1-2541-4bea-8eb7-262e808fa381 | Environment variables loaded successfully
2026-02-04 15:55:05 | INFO     | 81d104d1-2541-4bea-8eb7-262e808fa381 | Notion API Version: 2025-09-03
2026-02-04 15:55:05 | INFO     | 81d104d1-2541-4bea-8eb7-262e808fa381 | Database IDs configured: Papers, Events, Monitoring Targets, Monitoring Queue
2026-02-04 15:55:05 | INFO     | 81d104d1-2541-4bea-8eb7-262e808fa381 | Database schemas defined successfully
2026-02-04 15:55:05 | INFO     | 81d104d1-2541-4bea-8eb7-262e808fa381 | Schemas registered for: papers, events, monitoring_targets, monitoring_queue
2026-02-04 15:55:05 | INFO     | 81d104d1-2541-4bea-8eb7-262e808fa381 | Notion API base URL: https://api.notion.com/v1
2026-02-04 15:55:05 | INFO     | 81d104d1-2541-4bea-8eb7-262e808fa381 | Notion API version: 2025-09-03
2026-02-04 15:55:05 | INFO     | 81d104d1-2541-4bea-8eb7-262e808fa381 | NotionClient initi

In [39]:
# ============================================================
# Cell 05 — Execute 030_daily_paper_scanner with error handling
# ============================================================
# Overview:
#   Execute 030_daily_paper_scanner.ipynb to scan arXiv for new papers
#   matching research interests. This is an optional module; failures
#   are logged but do not halt the pipeline.
#
# Inputs / Outputs:
#   Inputs: MODULE_REGISTRY[1], orchestrator_state, globals from 028
#   Outputs: Updated orchestrator_state['module_results']['daily_paper_scanner'],
#            papers_found count captured from module globals
#
# Notes:
#   - Continues pipeline on failure (required=False)
#   - Captures papers_found metric from module exports
#   - Uses execute_notebook() and capture_module_metrics() from Cell 01
#   - Depends on 028 state (run_id, logger, config)
#

print('[Cell 05] Executing 030_daily_paper_scanner...')
print('=' * 60)

# Find module by name (order-independent)
paper_scanner_module = get_module("daily_paper_scanner")

# --- NEW: snapshot before execution (prevents "globals leftover" contamination)
_before = snapshot_keys()

# Execute the notebook
notebook_path = os.path.join(EXECUTION_CONFIG['notebooks_dir'], paper_scanner_module['notebook'])
result = execute_notebook(notebook_path, paper_scanner_module['name'])

# --- NEW: snapshot after execution + diff
_after = snapshot_keys()
if EXECUTION_CONFIG.get('capture_metrics', True):
    result['metrics'] = diff_metrics(_before, _after)
else:
    result['metrics'] = {}

# Store execution result
orchestrator_state['module_results'][paper_scanner_module['name']] = result

# Check execution status
if result['status'] == 'failed':
    error_msg = f"{paper_scanner_module['name']} failed (non-critical)"
    print(f"\n[WARNING] {error_msg}")
    print(f"Error details: {result.get('error', 'Unknown error')}")

    orchestrator_state['errors'].append({
        'module': paper_scanner_module['name'],
        'error': result.get('error'),
        'timestamp': result.get('end_time'),
        'critical': False
    })

    if not EXECUTION_CONFIG['continue_on_failure']:
        print('[ERROR] continue_on_failure=False, halting pipeline')
        orchestrator_state['pipeline_status'] = 'failed'
        raise RuntimeError(error_msg)
    else:
        print('[INFO] Continuing pipeline despite failure')

elif result['status'] == 'skipped':
    print(f"\n[INFO] {paper_scanner_module['name']} skipped")
    print(f"Reason: {result.get('error', 'Notebook not found')}")

else:
    print(f"\n[SUCCESS] {paper_scanner_module['name']} completed in {result['duration_seconds']}s")

    # Read metrics from result['metrics'] (diff-based)
    metrics = result.get('metrics') or {}
    papers_found = metrics.get('papers_found', 0)
    if isinstance(papers_found, int):
        print(f"  - Papers found: {papers_found}")

    if metrics:
        print(f"  - Captured metrics (diff): {list(metrics.keys())}")
    else:
        print("  - No metrics captured (diff empty; module may not set tracked globals)")

print('\n' + '=' * 60)
print('[Cell 05] 030_daily_paper_scanner execution completed.')
print(f"  - Status: {result['status']}")
print(f"  - Duration: {result['duration_seconds']}s")
if result['status'] == 'success' and result.get('metrics'):
    print(f"  - Metrics: {result['metrics']}")
print('=' * 60)


[Cell 05] Executing 030_daily_paper_scanner...
✓ Cell 01: Imports and dependencies loaded
✓ Cell 02: Environment bootstrap completed
  - NOTION_TOKEN: ntn_38...***
  - NOTION_VERSION: 2025-09-03
  - NOTION_LIT_DB_ID: set
  - NOTION_EVENTS_DB_ID: set
  - NOTION_MONITORING_TARGETS_DB_ID: set
  - NOTION_MONITORING_QUEUE_DB_ID: set
✓ Cell 03: Loaded configuration from config.yaml
  - Drive folder_id:      (set)
✓ Cell 03: Configuration validated (source: config.yaml)
  - Pipeline cadence:     daily
  - Max runtime:          30 min
  - Lookback (daily):     7 days
  - Lookback (tasks):     14 days
  - Lookback (projects):  30 days
  - Max items per query:  100
  - Logging level:        INFO
✓ Cell 04: Run context initialized
  - Run ID:           72735f50-93e2-4e5f-b6b4-6f704c106db2
  - Execution start:  2026-02-04T07:26:12.670341+00:00
  - Run date:         2026-02-04
  - Timezone:         UTC
  - Cadence:          daily
  - Lookback windows:
    - daily_notes : 7 days (2026-01-28 to 2026-

In [49]:
# ============================================================
# Cell 06 — Execute 031_pdf_inbox_processor with error handling (JSON-metrics preferred)
# ============================================================
# Overview:
#   Execute 031_pdf_inbox_processor.ipynb to process PDFs from the inbox.
#   Optional module; failures are logged but do not halt the pipeline.
#
# Inputs / Outputs:
#   Inputs: module registry entry for pdf_inbox_processor, orchestrator_state
#   Outputs: orchestrator_state['module_results']['pdf_inbox_processor'] with metrics
#
# Notes:
#   - Prefers JSON summary produced by 031 under artifacts/summaries/{run_id}_summary.json
#     (robust across separate kernels/processes).
#   - Falls back to diff_metrics(snapshot_keys) and then get_state if available.
#   - Persists extracted "pdfs_processed" into result['metrics'] for Cell 11 aggregation.
#

print('[Cell 06] Executing 031_pdf_inbox_processor...')
print('=' * 60)

import json
from pathlib import Path

def _safe_int(x, default=None):
    try:
        if x is None:
            return default
        if isinstance(x, bool):
            return int(x)
        if isinstance(x, (int, float)):
            return int(x)
        if isinstance(x, str) and x.strip().isdigit():
            return int(x.strip())
    except Exception:
        pass
    return default

def _load_summary_json(run_id: str):
    """
    Load summary JSON from artifacts/summaries/{run_id}_summary.json.
    If not found, fall back to newest *_summary.json (best-effort).
    """
    summaries_dir = Path("artifacts") / "summaries"
    candidate = summaries_dir / f"{run_id}_summary.json"

    try:
        if candidate.exists():
            return candidate, json.loads(candidate.read_text(encoding="utf-8"))
    except Exception:
        pass

    # Fallback: newest summary json (useful when run_id mismatches for any reason)
    try:
        if summaries_dir.exists():
            files = sorted(summaries_dir.glob("*_summary.json"), key=lambda p: p.stat().st_mtime, reverse=True)
            if files:
                p = files[0]
                return p, json.loads(p.read_text(encoding="utf-8"))
    except Exception:
        pass

    return None, None

# Find module by name (order-independent)
pdf_processor_module = get_module("pdf_inbox_processor")

# Snapshot before (diff fallback)
_before = snapshot_keys()

# Execute the notebook
notebook_path = os.path.join(EXECUTION_CONFIG['notebooks_dir'], pdf_processor_module['notebook'])
result = execute_notebook(notebook_path, pdf_processor_module['name'])

# Snapshot after (diff fallback)
_after = snapshot_keys()
diff = diff_metrics(_before, _after) if EXECUTION_CONFIG.get('capture_metrics', True) else {}

# Store execution result early
orchestrator_state['module_results'][pdf_processor_module['name']] = result

# Check execution status
if result['status'] == 'failed':
    error_msg = f"{pdf_processor_module['name']} failed (non-critical)"
    print(f"\n[WARNING] {error_msg}")
    print(f"Error details: {result.get('error', 'Unknown error')}")

    orchestrator_state['errors'].append({
        'module': pdf_processor_module['name'],
        'error': result.get('error', 'Unknown error'),
        'timestamp': result.get('end_time'),
        'critical': False
    })

    if not EXECUTION_CONFIG['continue_on_failure']:
        print('[ERROR] continue_on_failure=False, halting pipeline')
        orchestrator_state['pipeline_status'] = 'failed'
        raise RuntimeError(error_msg)
    else:
        print('[INFO] Continuing pipeline despite failure')

elif result['status'] == 'skipped':
    print(f"\n[INFO] {pdf_processor_module['name']} skipped")
    print(f"Reason: {result.get('error', 'Notebook not found')}")

else:
    print(f"\n[SUCCESS] {pdf_processor_module['name']} completed in {result.get('duration_seconds', 0.0)}s")

    rid = orchestrator_state.get("run_id") or globals().get("run_id")
    pdfs_processed = None
    metrics = {}

    # ----------------------------
    # A) Prefer JSON summary (cross-kernel robust)
    # ----------------------------
    if rid:
        p, summary = _load_summary_json(str(rid))
        if isinstance(summary, dict):
            # 031 writes: success_count, statistics.success, etc.
            pdfs_processed = _safe_int(summary.get("success_count"), default=None)
            if pdfs_processed is None:
                stats = summary.get("statistics", {}) or {}
                pdfs_processed = _safe_int(stats.get("success"), default=None)

            metrics["summary_json_used"] = True
            metrics["summary_json_path"] = str(p) if p else str(Path("artifacts") / "summaries" / f"{rid}_summary.json")

    # ----------------------------
    # B) Fallback to diff metrics (same-kernel only)
    # ----------------------------
    if pdfs_processed is None:
        metrics.update(diff or {})
        pdfs_processed = _safe_int(
            (metrics.get("pdfs_processed") if isinstance(metrics, dict) else None),
            default=None
        )
        if pdfs_processed is not None:
            metrics["summary_json_used"] = False

    # ----------------------------
    # C) Fallback to state getter (same-kernel only)
    # ----------------------------
    if pdfs_processed is None and callable(globals().get("get_state")):
        try:
            v = get_state("pdfs_processed")
            pdfs_processed = _safe_int(v, default=None)
            if pdfs_processed is not None:
                metrics["summary_json_used"] = False
                metrics["state_fallback_used"] = True
        except Exception:
            pass

    pdfs_processed = pdfs_processed if isinstance(pdfs_processed, int) else 0
    print(f"  - PDFs processed: {pdfs_processed}")

    # Attach metrics for downstream aggregation (Cell 11)
    result['metrics'] = metrics or {}
    result['metrics']['pdfs_processed'] = pdfs_processed

    if result['metrics']:
        print(f"  - Captured metrics keys: {list(result['metrics'].keys())}")
    else:
        print("  - No metrics captured")

print('\n' + '=' * 60)
print('[Cell 06] 031_pdf_inbox_processor execution completed.')
print(f"  - Status: {result['status']}")
print(f"  - Duration: {result.get('duration_seconds', 0.0)}s")
if result.get('metrics'):
    print(f"  - Metrics: {result['metrics']}")
print('=' * 60)


[Cell 06] Executing 031_pdf_inbox_processor...
2026-02-04 16:54:16 | INFO     | c034f05d-b51b-4469-a027-30552ab4403e | file_cache is only supported with oauth2client<4.0.0
✅ Google Drive service initialized and ready
✓ Cell 01: Imports and dependencies loaded
✓ Cell 02: Environment bootstrap completed
  - NOTION_TOKEN: ntn_38...***
  - NOTION_VERSION: 2025-09-03
  - NOTION_LIT_DB_ID: set
  - NOTION_EVENTS_DB_ID: set
  - NOTION_MONITORING_TARGETS_DB_ID: set
  - NOTION_MONITORING_QUEUE_DB_ID: set
✓ Cell 03: Loaded configuration from config.yaml
  - Drive folder_id:      (set)
✓ Cell 03: Configuration validated (source: config.yaml)
  - Pipeline cadence:     daily
  - Max runtime:          30 min
  - Lookback (daily):     7 days
  - Lookback (tasks):     14 days
  - Lookback (projects):  30 days
  - Max items per query:  100
  - Logging level:        INFO
✓ Cell 04: Run context initialized
  - Run ID:           a4097c27-1d3f-4263-8a5a-02b5c5e1d12d
  - Execution start:  2026-02-04T07:54:16

In [41]:
# ============================================================
# Cell 07 — Execute 032_monitor_vc_daily with error handling (diff-metrics)
# ============================================================
# Overview:
#   Execute 032_monitor_vc_daily.ipynb to monitor VC firms for new updates.
#   Optional module; failures are logged but do not halt the pipeline.
#
# Inputs / Outputs:
#   Inputs: module registry entry for monitor_vc_daily, orchestrator_state
#   Outputs: orchestrator_state['module_results']['monitor_vc_daily'] with diff-based metrics
#
# Notes:
#   - Uses snapshot_keys() + diff_metrics() (defined in Cell 01) to avoid globals contamination
#   - Continues pipeline on failure (required=False)
#

print('[Cell 07] Executing 032_monitor_vc_daily...')
print('=' * 60)

# Find module by name (order-independent)
vc_monitor_module = get_module("monitor_vc_daily")

# --- NEW: snapshot before execution
_before = snapshot_keys()

# Execute the notebook
notebook_path = os.path.join(EXECUTION_CONFIG['notebooks_dir'], vc_monitor_module['notebook'])
result = execute_notebook(notebook_path, vc_monitor_module['name'])

# --- NEW: snapshot after execution + diff
_after = snapshot_keys()
if EXECUTION_CONFIG.get('capture_metrics', True):
    result['metrics'] = diff_metrics(_before, _after)
else:
    result['metrics'] = {}

# Store execution result
orchestrator_state['module_results'][vc_monitor_module['name']] = result

# Check execution status
if result['status'] == 'failed':
    error_msg = f"{vc_monitor_module['name']} failed (non-critical)"
    print(f"\n[WARNING] {error_msg}")
    print(f"Error details: {result.get('error', 'Unknown error')}")

    orchestrator_state['errors'].append({
        'module': vc_monitor_module['name'],
        'error': result.get('error'),
        'timestamp': result.get('end_time'),
        'critical': False
    })

    if not EXECUTION_CONFIG['continue_on_failure']:
        print('[ERROR] continue_on_failure=False, halting pipeline')
        orchestrator_state['pipeline_status'] = 'failed'
        raise RuntimeError(error_msg)
    else:
        print('[INFO] Continuing pipeline despite failure')

elif result['status'] == 'skipped':
    print(f"\n[INFO] {vc_monitor_module['name']} skipped")
    print(f"Reason: {result.get('error', 'Notebook not found')}")

else:
    print(f"\n[SUCCESS] {vc_monitor_module['name']} completed in {result['duration_seconds']}s")

    # Read diff-based metrics
    metrics = result.get('metrics') or {}

    # Extract updates count if available (common keys)
    updates_found = (
        metrics.get('updates_found')
        or metrics.get('events_created')
        or metrics.get('created_events')
        or 0
    )
    if isinstance(updates_found, int):
        print(f"  - VC updates found: {updates_found}")

    if metrics:
        print(f"  - Captured metrics (diff): {list(metrics.keys())}")
    else:
        print("  - No metrics captured (diff empty; module may not set tracked globals)")

print('\n' + '=' * 60)
print('[Cell 07] 032_monitor_vc_daily execution completed.')
print(f"  - Status: {result['status']}")
print(f"  - Duration: {result['duration_seconds']}s")
if result['status'] == 'success' and result.get('metrics'):
    print(f"  - Metrics: {result['metrics']}")
print('=' * 60)


[Cell 07] Executing 032_monitor_vc_daily...
✓ Cell 01: Imports and dependencies loaded
✓ Cell 02: Environment bootstrap completed
  - NOTION_TOKEN: ntn_38...***
  - NOTION_VERSION: 2025-09-03
  - NOTION_LIT_DB_ID: set
  - NOTION_EVENTS_DB_ID: set
  - NOTION_MONITORING_TARGETS_DB_ID: set
  - NOTION_MONITORING_QUEUE_DB_ID: set
✓ Cell 03: Loaded configuration from config.yaml
  - Drive folder_id:      (set)
✓ Cell 03: Configuration validated (source: config.yaml)
  - Pipeline cadence:     daily
  - Max runtime:          30 min
  - Lookback (daily):     7 days
  - Lookback (tasks):     14 days
  - Lookback (projects):  30 days
  - Max items per query:  100
  - Logging level:        INFO
✓ Cell 04: Run context initialized
  - Run ID:           c03b21b8-b75a-4fd0-86ae-883a10409c1d
  - Execution start:  2026-02-04T07:30:50.774488+00:00
  - Run date:         2026-02-04
  - Timezone:         UTC
  - Cadence:          daily
  - Lookback windows:
    - daily_notes : 7 days (2026-01-28 to 2026-02-

In [42]:
# ============================================================
# Cell 08 — Execute 033_monitor_startups_daily with error handling (diff-metrics)
# ============================================================
# Overview:
#   Execute 033_monitor_startups_daily.ipynb to monitor startup companies for new updates.
#   Optional module; failures are logged but do not halt the pipeline.
#
# Inputs / Outputs:
#   Inputs: module registry entry for monitor_startups_daily, orchestrator_state
#   Outputs: orchestrator_state['module_results']['monitor_startups_daily'] with diff-based metrics
#
# Notes:
#   - Uses snapshot_keys() + diff_metrics() (defined in Cell 01) to avoid globals contamination
#   - Continues pipeline on failure (required=False)
#

print('[Cell 08] Executing 033_monitor_startups_daily...')
print('=' * 60)

# Find module by name (order-independent)
startups_monitor_module = get_module("monitor_startups_daily")

# --- NEW: snapshot before execution
_before = snapshot_keys()

# Execute the notebook
notebook_path = os.path.join(EXECUTION_CONFIG['notebooks_dir'], startups_monitor_module['notebook'])
result = execute_notebook(notebook_path, startups_monitor_module['name'])

# --- NEW: snapshot after execution + diff
_after = snapshot_keys()
if EXECUTION_CONFIG.get('capture_metrics', True):
    result['metrics'] = diff_metrics(_before, _after)
else:
    result['metrics'] = {}

# Store execution result
orchestrator_state['module_results'][startups_monitor_module['name']] = result

# Check execution status
if result['status'] == 'failed':
    error_msg = f"{startups_monitor_module['name']} failed (non-critical)"
    print(f"\n[WARNING] {error_msg}")
    print(f"Error details: {result.get('error', 'Unknown error')}")

    orchestrator_state['errors'].append({
        'module': startups_monitor_module['name'],
        'error': result.get('error'),
        'timestamp': result.get('end_time'),
        'critical': False
    })

    if not EXECUTION_CONFIG['continue_on_failure']:
        print('[ERROR] continue_on_failure=False, halting pipeline')
        orchestrator_state['pipeline_status'] = 'failed'
        raise RuntimeError(error_msg)
    else:
        print('[INFO] Continuing pipeline despite failure')

elif result['status'] == 'skipped':
    print(f"\n[INFO] {startups_monitor_module['name']} skipped")
    print(f"Reason: {result.get('error', 'Notebook not found')}")

else:
    print(f"\n[SUCCESS] {startups_monitor_module['name']} completed in {result['duration_seconds']}s")

    # Read diff-based metrics
    metrics = result.get('metrics') or {}

    # Extract updates count if available (common keys)
    updates_found = (
        metrics.get('updates_found')
        or metrics.get('events_created')
        or metrics.get('created_events')
        or 0
    )
    if isinstance(updates_found, int):
        print(f"  - Startup updates found: {updates_found}")

    if metrics:
        print(f"  - Captured metrics (diff): {list(metrics.keys())}")
    else:
        print("  - No metrics captured (diff empty; module may not set tracked globals)")

print('\n' + '=' * 60)
print('[Cell 08] 033_monitor_startups_daily execution completed.')
print(f"  - Status: {result['status']}")
print(f"  - Duration: {result['duration_seconds']}s")
if result['status'] == 'success' and result.get('metrics'):
    print(f"  - Metrics: {result['metrics']}")
print('=' * 60)


[Cell 08] Executing 033_monitor_startups_daily...
✓ Cell 01: Imports and dependencies loaded
✓ Cell 02: Environment bootstrap completed
  - NOTION_TOKEN: ntn_38...***
  - NOTION_VERSION: 2025-09-03
  - NOTION_LIT_DB_ID: set
  - NOTION_EVENTS_DB_ID: set
  - NOTION_MONITORING_TARGETS_DB_ID: set
  - NOTION_MONITORING_QUEUE_DB_ID: set
✓ Cell 03: Loaded configuration from config.yaml
  - Drive folder_id:      (set)
✓ Cell 03: Configuration validated (source: config.yaml)
  - Pipeline cadence:     daily
  - Max runtime:          30 min
  - Lookback (daily):     7 days
  - Lookback (tasks):     14 days
  - Lookback (projects):  30 days
  - Max items per query:  100
  - Logging level:        INFO
✓ Cell 04: Run context initialized
  - Run ID:           2c0a89d4-9292-46cb-afbb-656104095d2b
  - Execution start:  2026-02-04T07:31:55.238009+00:00
  - Run date:         2026-02-04
  - Timezone:         UTC
  - Cadence:          daily
  - Lookback windows:
    - daily_notes : 7 days (2026-01-28 to 20

In [44]:
# ============================================================
# Cell 09 — Execute 034_monitor_policy_daily with error handling (diff-metrics)
# ============================================================
# Overview:
#   Execute 034_monitor_policy_daily.ipynb to monitor policy sources for new updates.
#   Optional module; failures are logged but do not halt the pipeline.
#
# Inputs / Outputs:
#   Inputs: module registry entry for monitor_policy_daily, orchestrator_state
#   Outputs: orchestrator_state['module_results']['monitor_policy_daily'] with diff-based metrics
#
# Notes:
#   - Uses snapshot_keys() + diff_metrics() (defined in Cell 01) to avoid globals contamination
#   - Continues pipeline on failure (required=False)
#

print('[Cell 09] Executing 034_monitor_policy_daily...')
print('=' * 60)

# Find module by name (order-independent)
policy_monitor_module = get_module("monitor_policy_daily")

# --- NEW: snapshot before execution
_before = snapshot_keys()

# Execute the notebook
notebook_path = os.path.join(EXECUTION_CONFIG['notebooks_dir'], policy_monitor_module['notebook'])
result = execute_notebook(notebook_path, policy_monitor_module['name'])

# --- NEW: snapshot after execution + diff
_after = snapshot_keys()
if EXECUTION_CONFIG.get('capture_metrics', True):
    result['metrics'] = diff_metrics(_before, _after)
else:
    result['metrics'] = {}

# Store execution result
orchestrator_state['module_results'][policy_monitor_module['name']] = result

# Check execution status
if result['status'] == 'failed':
    error_msg = f"{policy_monitor_module['name']} failed (non-critical)"
    print(f"\n[WARNING] {error_msg}")
    print(f"Error details: {result.get('error', 'Unknown error')}")

    orchestrator_state['errors'].append({
        'module': policy_monitor_module['name'],
        'error': result.get('error'),
        'timestamp': result.get('end_time'),
        'critical': False
    })

    if not EXECUTION_CONFIG['continue_on_failure']:
        print('[ERROR] continue_on_failure=False, halting pipeline')
        orchestrator_state['pipeline_status'] = 'failed'
        raise RuntimeError(error_msg)
    else:
        print('[INFO] Continuing pipeline despite failure')

elif result['status'] == 'skipped':
    print(f"\n[INFO] {policy_monitor_module['name']} skipped")
    print(f"Reason: {result.get('error', 'Notebook not found')}")

else:
    print(f"\n[SUCCESS] {policy_monitor_module['name']} completed in {result['duration_seconds']}s")

    # Read diff-based metrics
    metrics = result.get('metrics') or {}

    # Extract updates count if available (common keys)
    updates_found = (
        metrics.get('updates_found')
        or metrics.get('events_created')
        or metrics.get('created_events')
        or 0
    )
    if isinstance(updates_found, int):
        print(f"  - Policy updates found: {updates_found}")

    if metrics:
        print(f"  - Captured metrics (diff): {list(metrics.keys())}")
    else:
        print("  - No metrics captured (diff empty; module may not set tracked globals)")

print('\n' + '=' * 60)
print('[Cell 09] 034_monitor_policy_daily execution completed.')
print(f"  - Status: {result['status']}")
print(f"  - Duration: {result['duration_seconds']}s")
if result['status'] == 'success' and result.get('metrics'):
    print(f"  - Metrics: {result['metrics']}")
print('=' * 60)


[Cell 09] Executing 034_monitor_policy_daily...
✓ Cell 01: Imports and dependencies loaded
✓ Cell 02: Environment bootstrap completed
  - NOTION_TOKEN: ntn_38...***
  - NOTION_VERSION: 2025-09-03
  - NOTION_LIT_DB_ID: set
  - NOTION_EVENTS_DB_ID: set
  - NOTION_MONITORING_TARGETS_DB_ID: set
  - NOTION_MONITORING_QUEUE_DB_ID: set
✓ Cell 03: Loaded configuration from config.yaml
  - Drive folder_id:      (set)
✓ Cell 03: Configuration validated (source: config.yaml)
  - Pipeline cadence:     daily
  - Max runtime:          30 min
  - Lookback (daily):     7 days
  - Lookback (tasks):     14 days
  - Lookback (projects):  30 days
  - Max items per query:  100
  - Logging level:        INFO
✓ Cell 04: Run context initialized
  - Run ID:           3b1c0775-83e8-43b2-955a-e7a5db8e5276
  - Execution start:  2026-02-04T07:33:00.103915+00:00
  - Run date:         2026-02-04
  - Timezone:         UTC
  - Cadence:          daily
  - Lookback windows:
    - daily_notes : 7 days (2026-01-28 to 2026

In [45]:
# ============================================================
# Cell 10 — Execute 035_monitor_people_daily with error handling (diff-metrics)
# ============================================================
# Overview:
#   Execute 035_monitor_people_daily.ipynb to monitor people and thought leaders for new updates.
#   Optional module; failures are logged but do not halt the pipeline.
#
# Inputs / Outputs:
#   Inputs: module registry entry for monitor_people_daily, orchestrator_state
#   Outputs: orchestrator_state['module_results']['monitor_people_daily'] with diff-based metrics
#
# Notes:
#   - Uses snapshot_keys() + diff_metrics() (defined in Cell 01) to avoid globals contamination
#   - Continues pipeline on failure (required=False)
#   - Final monitor module before metric aggregation in Cell 11
#

print('[Cell 10] Executing 035_monitor_people_daily...')
print('=' * 60)

# Find module by name (order-independent)
people_monitor_module = get_module("monitor_people_daily")

# --- NEW: snapshot before execution
_before = snapshot_keys()

# Execute the notebook
notebook_path = os.path.join(EXECUTION_CONFIG['notebooks_dir'], people_monitor_module['notebook'])
result = execute_notebook(notebook_path, people_monitor_module['name'])

# --- NEW: snapshot after execution + diff
_after = snapshot_keys()
if EXECUTION_CONFIG.get('capture_metrics', True):
    result['metrics'] = diff_metrics(_before, _after)
else:
    result['metrics'] = {}

# Store execution result
orchestrator_state['module_results'][people_monitor_module['name']] = result

# Check execution status
if result['status'] == 'failed':
    error_msg = f"{people_monitor_module['name']} failed (non-critical)"
    print(f"\n[WARNING] {error_msg}")
    print(f"Error details: {result.get('error', 'Unknown error')}")

    orchestrator_state['errors'].append({
        'module': people_monitor_module['name'],
        'error': result.get('error'),
        'timestamp': result.get('end_time'),
        'critical': False
    })

    if not EXECUTION_CONFIG['continue_on_failure']:
        print('[ERROR] continue_on_failure=False, halting pipeline')
        orchestrator_state['pipeline_status'] = 'failed'
        raise RuntimeError(error_msg)
    else:
        print('[INFO] Continuing pipeline despite failure')

elif result['status'] == 'skipped':
    print(f"\n[INFO] {people_monitor_module['name']} skipped")
    print(f"Reason: {result.get('error', 'Notebook not found')}")

else:
    print(f"\n[SUCCESS] {people_monitor_module['name']} completed in {result['duration_seconds']}s")

    # Read diff-based metrics
    metrics = result.get('metrics') or {}

    # Extract updates count if available (common keys)
    updates_found = (
        metrics.get('updates_found')
        or metrics.get('events_created')
        or metrics.get('created_events')
        or 0
    )
    if isinstance(updates_found, int):
        print(f"  - People updates found: {updates_found}")

    if metrics:
        print(f"  - Captured metrics (diff): {list(metrics.keys())}")
    else:
        print("  - No metrics captured (diff empty; module may not set tracked globals)")

print('\n' + '=' * 60)
print('[Cell 10] 035_monitor_people_daily execution completed.')
print(f"  - Status: {result['status']}")
print(f"  - Duration: {result['duration_seconds']}s")
if result['status'] == 'success' and result.get('metrics'):
    print(f"  - Metrics: {result['metrics']}")
print('=' * 60)
print('\n[INFO] All component modules executed. Ready for metric aggregation.')


[Cell 10] Executing 035_monitor_people_daily...
✓ Cell 01: Imports and dependencies loaded
✓ Cell 02: Environment bootstrap completed
  - NOTION_TOKEN: ntn_38...***
  - NOTION_VERSION: 2025-09-03
  - NOTION_LIT_DB_ID: set
  - NOTION_EVENTS_DB_ID: set
  - NOTION_MONITORING_TARGETS_DB_ID: set
  - NOTION_MONITORING_QUEUE_DB_ID: set
✓ Cell 03: Loaded configuration from config.yaml
  - Drive folder_id:      (set)
✓ Cell 03: Configuration validated (source: config.yaml)
  - Pipeline cadence:     daily
  - Max runtime:          30 min
  - Lookback (daily):     7 days
  - Lookback (tasks):     14 days
  - Lookback (projects):  30 days
  - Max items per query:  100
  - Logging level:        INFO
✓ Cell 04: Run context initialized
  - Run ID:           46350e87-85ef-4643-b871-fdc12e8c104b
  - Execution start:  2026-02-04T07:35:47.459012+00:00
  - Run date:         2026-02-04
  - Timezone:         UTC
  - Cadence:          daily
  - Lookback windows:
    - daily_notes : 7 days (2026-01-28 to 2026

In [50]:
# ============================================================
# Cell 11 — Aggregate metrics from all module executions (diff-metrics aware)
# ============================================================
# Overview:
#   Aggregate execution metrics from all module results into orchestrator_state['aggregated_metrics'].
#   This version is compatible with diff-based metrics (snapshot_keys + diff_metrics).
#
# Notes:
#   - Prefers explicit top-level numeric metrics (papers_found/pdf.../updates_found)
#   - Falls back to run_stats / scan_stats dictionaries if present
#   - Never assumes metric keys exist; uses defensive access
#

print('[Cell 11] Aggregating metrics from all module executions...')
print('=' * 60)

def _as_int(x, default=0):
    try:
        if x is None:
            return default
        if isinstance(x, bool):
            return int(x)
        if isinstance(x, (int,)):
            return int(x)
        if isinstance(x, float):
            return int(x)
        if isinstance(x, str) and x.strip().isdigit():
            return int(x.strip())
    except Exception:
        pass
    return default

def _pick_first_int(d: dict, keys: list[str], default=0):
    if not isinstance(d, dict):
        return default
    for k in keys:
        v = d.get(k)
        if isinstance(v, bool):
            return int(v)
        if isinstance(v, (int, float)):
            return int(v)
        if isinstance(v, str) and v.strip().isdigit():
            return int(v.strip())
    return default

def _extract_from_nested_stats(metrics: dict, keys: list[str], default=0):
    """
    Try to extract count from metrics['run_stats'] or metrics['scan_stats'] if they exist.
    """
    if not isinstance(metrics, dict):
        return default
    for container_key in ("run_stats", "scan_stats"):
        container = metrics.get(container_key)
        if isinstance(container, dict):
            v = _pick_first_int(container, keys, default=None)
            if v is not None:
                return v
    return default

# Record pipeline end time
pipeline_end_time = datetime.now(timezone.utc)
orchestrator_state['pipeline_end'] = pipeline_end_time.isoformat()
orchestrator_state['pipeline_duration_seconds'] = round(
    (pipeline_end_time - pipeline_start_time).total_seconds(), 2
)

# Initialize aggregated metrics dict (ensure exists)
orchestrator_state.setdefault('aggregated_metrics', {})
agg = orchestrator_state['aggregated_metrics']

# Count module execution statuses
module_results = orchestrator_state.get('module_results', {})
agg['total_modules_executed'] = len(module_results)
agg['modules_succeeded'] = sum(1 for r in module_results.values() if r.get('status') == 'success')
agg['modules_failed'] = sum(1 for r in module_results.values() if r.get('status') == 'failed')
agg['modules_skipped'] = sum(1 for r in module_results.values() if r.get('status') == 'skipped')

# Sum total execution time across all modules
agg['total_execution_time'] = round(
    sum(float(r.get('duration_seconds', 0.0) or 0.0) for r in module_results.values()), 2
)

print(f'\nModule execution summary:')
print(f"  - Total modules: {agg['total_modules_executed']}")
print(f"  - Succeeded: {agg['modules_succeeded']}")
print(f"  - Failed: {agg['modules_failed']}")
print(f"  - Skipped: {agg['modules_skipped']}")
print(f"  - Total execution time: {agg['total_execution_time']}s")

# --------------------------
# Extract module-specific metrics (diff-metrics aware)
# --------------------------

# 030_daily_paper_scanner
paper_scanner_metrics = (module_results.get('daily_paper_scanner') or {}).get('metrics') or {}
agg['papers_found'] = _pick_first_int(
    paper_scanner_metrics,
    keys=['papers_found', 'papers_created', 'new_papers', 'n_papers'],
    default=0
)
if agg['papers_found'] == 0:
    agg['papers_found'] = _extract_from_nested_stats(
        paper_scanner_metrics,
        keys=['papers_found', 'papers_created', 'created', 'new_records'],
        default=0
    )

# 031_pdf_inbox_processor
pdf_processor_metrics = (module_results.get('pdf_inbox_processor') or {}).get('metrics') or {}
agg['pdfs_processed'] = _pick_first_int(
    pdf_processor_metrics,
    keys=['pdfs_processed', 'processed', 'processed_count', 'n_processed'],
    default=0
)
if agg['pdfs_processed'] == 0:
    agg['pdfs_processed'] = _extract_from_nested_stats(
        pdf_processor_metrics,
        keys=['pdfs_processed', 'processed', 'processed_count', 'n_processed'],
        default=0
    )

def _extract_updates(module_name: str) -> int:
    m = (module_results.get(module_name) or {}).get('metrics') or {}
    v = _pick_first_int(
        m,
        keys=['updates_found', 'events_created', 'created_events', 'n_created', 'created'],
        default=0
    )
    if v == 0:
        # try nested stats
        v = _extract_from_nested_stats(
            m,
            keys=['updates_found', 'events_created', 'created_events', 'n_created', 'created'],
            default=0
        )
    return v

agg['vc_updates'] = _extract_updates('monitor_vc_daily')
agg['startup_updates'] = _extract_updates('monitor_startups_daily')
agg['policy_updates'] = _extract_updates('monitor_policy_daily')
agg['people_updates'] = _extract_updates('monitor_people_daily')

# Total events created (pipeline-level)
agg['events_created'] = (
    _as_int(agg.get('papers_found')) +
    _as_int(agg.get('pdfs_processed')) +
    _as_int(agg.get('vc_updates')) +
    _as_int(agg.get('startup_updates')) +
    _as_int(agg.get('policy_updates')) +
    _as_int(agg.get('people_updates'))
)

print(f'\nModule-specific metrics:')
print(f"  - Papers found: {agg.get('papers_found', 0)}")
print(f"  - PDFs processed: {agg.get('pdfs_processed', 0)}")
print(f"  - VC updates: {agg.get('vc_updates', 0)}")
print(f"  - Startup updates: {agg.get('startup_updates', 0)}")
print(f"  - Policy updates: {agg.get('policy_updates', 0)}")
print(f"  - People updates: {agg.get('people_updates', 0)}")
print(f"  - Total events created: {agg.get('events_created', 0)}")

# Determine pipeline status
if orchestrator_state.get('pipeline_status') != 'failed':
    orchestrator_state['pipeline_status'] = 'completed_with_failures' if agg['modules_failed'] > 0 else 'completed'

print(f"\nPipeline status: {orchestrator_state['pipeline_status']}")
print(f"Pipeline duration: {orchestrator_state['pipeline_duration_seconds']}s")

if orchestrator_state.get('errors'):
    print(f"\nErrors encountered: {len(orchestrator_state['errors'])}")
    for err in orchestrator_state['errors']:
        critical = ' [CRITICAL]' if err.get('critical', False) else ''
        print(f"  - {err.get('module')}{critical}: {err.get('error')}")

print('\n' + '=' * 60)
print('[Cell 11] Metric aggregation completed.')
print(f"  - Total events: {agg.get('events_created', 0)}")
print(f"  - Pipeline status: {orchestrator_state['pipeline_status']}")
print('=' * 60)


[Cell 11] Aggregating metrics from all module executions...

Module execution summary:
  - Total modules: 8
  - Succeeded: 8
  - Failed: 0
  - Skipped: 0
  - Total execution time: 436.32s

Module-specific metrics:
  - Papers found: 0
  - PDFs processed: 1
  - VC updates: 0
  - Startup updates: 0
  - Policy updates: 0
  - People updates: 1
  - Total events created: 2

Pipeline status: completed
Pipeline duration: 3626.26s

[Cell 11] Metric aggregation completed.
  - Total events: 2
  - Pipeline status: completed


In [51]:
# ============================================================
# Cell 12 — Generate Notion-ready Markdown summary
# ============================================================
# Overview:
#   Generate a Notion-optimized Markdown summary of the daily pipeline run.
#   Uses Notion-compatible formatting (callouts, toggles, tables, dividers).
#   Includes run metadata, module execution status, aggregated metrics,
#   per-module details, and error summaries.
#
# Inputs / Outputs:
#   Inputs: orchestrator_state (with module_results, aggregated_metrics, errors)
#   Outputs: orchestrator_state['summaries']['notion_markdown'] (str)
#
# Notes:
#   - Notion-specific formatting: > for callouts, --- for dividers
#   - Tables use standard Markdown table syntax (supported in Notion)
#   - Emoji used for visual hierarchy and status indicators
#   - Toggle blocks simulated with collapsible section headers
#   - Stored in orchestrator_state for export in Cell 15
#

print('[Cell 12] Generating Notion-ready Markdown summary...')
print('=' * 60)

# Helper: format duration as human-readable string
def format_duration(seconds: float) -> str:
    """Convert seconds to human-readable duration (e.g., '2m 34s')."""
    if seconds < 60:
        return f"{seconds:.1f}s"
    minutes = int(seconds // 60)
    secs = seconds % 60
    return f"{minutes}m {secs:.1f}s"

# Helper: format timestamp for display
def format_timestamp(iso_timestamp: str) -> str:
    """Format ISO timestamp as readable UTC string."""
    if not iso_timestamp:
        return 'N/A'
    try:
        dt = datetime.fromisoformat(iso_timestamp.replace('Z', '+00:00'))
        return dt.strftime('%Y-%m-%d %H:%M:%S UTC')
    except:
        return iso_timestamp

# Helper: status emoji
def status_emoji(status: str) -> str:
    """Return emoji for module execution status."""
    return {
        'success': '✅',
        'failed': '❌',
        'skipped': '⏭️',
        'running': '🔄',
        'completed': '✅',
        'completed_with_failures': '⚠️'
    }.get(status, '❓')

# Extract orchestrator state
run_id = orchestrator_state.get('run_id', 'unknown')
pipeline_status = orchestrator_state.get('pipeline_status', 'unknown')
pipeline_start = orchestrator_state.get('pipeline_start')
pipeline_end = orchestrator_state.get('pipeline_end')
pipeline_duration = orchestrator_state.get('pipeline_duration_seconds', 0.0)
agg = orchestrator_state.get('aggregated_metrics', {})
module_results = orchestrator_state.get('module_results', {})
errors = orchestrator_state.get('errors', [])

# --- Build Notion-ready Markdown ---
notion_md = []

# Title and header
notion_md.append(f"# 📊 ResearchOS Daily Pipeline Summary\n")
notion_md.append(f"**Run ID:** `{run_id}`\n")
notion_md.append(f"**Status:** {status_emoji(pipeline_status)} {pipeline_status.replace('_', ' ').title()}\n")
notion_md.append(f"**Duration:** {format_duration(pipeline_duration)}\n")
notion_md.append("\n---\n\n")

# Pipeline metadata callout
notion_md.append("> 📅 **Pipeline Execution Metadata**\n")
notion_md.append(f"> - **Started:** {format_timestamp(pipeline_start)}\n")
notion_md.append(f"> - **Ended:** {format_timestamp(pipeline_end)}\n")
notion_md.append(f"> - **Total Duration:** {format_duration(pipeline_duration)}\n")
notion_md.append(f"> - **Modules Executed:** {agg.get('total_modules_executed', 0)}\n")
notion_md.append("\n---\n\n")

# Aggregated metrics summary
notion_md.append("## 📈 Aggregated Metrics\n\n")
notion_md.append("| Metric | Count |\n")
notion_md.append("|--------|-------|\n")
notion_md.append(f"| **Papers Found** | {agg.get('papers_found', 0)} |\n")
notion_md.append(f"| **PDFs Processed** | {agg.get('pdfs_processed', 0)} |\n")
notion_md.append(f"| **VC Updates** | {agg.get('vc_updates', 0)} |\n")
notion_md.append(f"| **Startup Updates** | {agg.get('startup_updates', 0)} |\n")
notion_md.append(f"| **Policy Updates** | {agg.get('policy_updates', 0)} |\n")
notion_md.append(f"| **People Updates** | {agg.get('people_updates', 0)} |\n")
notion_md.append(f"| **Total Events Created** | {agg.get('events_created', 0)} |\n")
notion_md.append("\n---\n\n")

# Module execution summary
notion_md.append("## 🔧 Module Execution Summary\n\n")
notion_md.append("| Module | Status | Duration | Details |\n")
notion_md.append("|--------|--------|----------|---------|\n")

for module_name, result in module_results.items():
    status = result.get('status', 'unknown')
    emoji = status_emoji(status)
    duration = format_duration(result.get('duration_seconds', 0.0))
    
    # Extract key metric if available
    details = '—'
    if result.get('metrics'):
        metrics = result['metrics']
        if 'papers_found' in metrics:
            details = f"{metrics['papers_found']} papers"
        elif 'pdfs_processed' in metrics:
            details = f"{metrics['pdfs_processed']} PDFs"
        elif 'updates_found' in metrics:
            details = f"{metrics['updates_found']} updates"
        elif 'events_created' in metrics:
            details = f"{metrics['events_created']} events"
    
    # Error indicator
    error_note = ''
    if status == 'failed':
        error_note = ' ⚠️ See errors below'
    
    notion_md.append(f"| **{module_name}** | {emoji} {status} | {duration} | {details}{error_note} |\n")

notion_md.append("\n---\n\n")

# Execution timeline callout
notion_md.append("> ⏱️ **Execution Timeline**\n")
notion_md.append(f"> - **Total Execution Time:** {format_duration(agg.get('total_execution_time', 0.0))}\n")
notion_md.append(f"> - **Pipeline Overhead:** {format_duration(pipeline_duration - agg.get('total_execution_time', 0.0))}\n")
notion_md.append("\n---\n\n")

# Module details (collapsible section)
notion_md.append("## 📋 Module Details\n\n")

for module_name, result in module_results.items():
    status = result.get('status', 'unknown')
    emoji = status_emoji(status)
    
    notion_md.append(f"### {emoji} {module_name}\n\n")
    notion_md.append(f"- **Status:** {status}\n")
    notion_md.append(f"- **Notebook:** `{result.get('notebook', 'N/A')}`\n")
    notion_md.append(f"- **Duration:** {format_duration(result.get('duration_seconds', 0.0))}\n")
    notion_md.append(f"- **Started:** {format_timestamp(result.get('start_time'))}\n")
    notion_md.append(f"- **Ended:** {format_timestamp(result.get('end_time'))}\n")
    
    # Metrics if available
    if result.get('metrics'):
        notion_md.append("- **Metrics:**\n")
        for key, value in result['metrics'].items():
            if isinstance(value, (int, str, float)):
                notion_md.append(f"  - `{key}`: {value}\n")
    
    # Error details if failed
    if status == 'failed' and result.get('error'):
        notion_md.append(f"- **Error:** `{result['error']}`\n")
    
    notion_md.append("\n")

notion_md.append("---\n\n")

# Error summary (if any)
if errors:
    notion_md.append("## ⚠️ Errors Encountered\n\n")
    notion_md.append("> **Critical errors may indicate pipeline failures.**\n\n")
    
    for i, error in enumerate(errors, 1):
        critical_flag = ' [CRITICAL]' if error.get('critical', False) else ''
        notion_md.append(f"{i}. **{error.get('module', 'Unknown')}{critical_flag}**\n")
        notion_md.append(f"   - **Error:** `{error.get('error', 'No details')}`\n")
        notion_md.append(f"   - **Timestamp:** {format_timestamp(error.get('timestamp'))}\n")
        notion_md.append("\n")
    
    notion_md.append("---\n\n")
else:
    notion_md.append("> ✅ **No errors encountered during this pipeline run.**\n\n")
    notion_md.append("---\n\n")

# Footer
notion_md.append("## 🏁 Pipeline Complete\n\n")
notion_md.append(f"{status_emoji(pipeline_status)} Pipeline finished with status: **{pipeline_status.replace('_', ' ').title()}**\n\n")
notion_md.append(f"**Next Steps:**\n")
notion_md.append("- Review Event entries in Notion\n")
notion_md.append("- Check failed modules (if any) for errors\n")
notion_md.append("- Verify data quality in outputs\n\n")
notion_md.append("---\n\n")
notion_md.append(f"*Generated: {format_timestamp(pipeline_end)}*\n")

# Join and store
notion_markdown = ''.join(notion_md)
orchestrator_state['summaries']['notion_markdown'] = notion_markdown

print(f'[SUCCESS] Notion-ready Markdown summary generated')
print(f'  - Length: {len(notion_markdown)} characters')
print(f'  - Sections: Pipeline metadata, metrics, module details, errors')
print(f'  - Format: Notion-compatible Markdown with callouts and tables')
print('=' * 60)

# Preview first few lines
preview_lines = notion_markdown.split('\n')[:15]
print('\nPreview (first 15 lines):')
print('\n'.join(preview_lines))
print('...')


[Cell 12] Generating Notion-ready Markdown summary...
[SUCCESS] Notion-ready Markdown summary generated
  - Length: 3469 characters
  - Sections: Pipeline metadata, metrics, module details, errors
  - Format: Notion-compatible Markdown with callouts and tables

Preview (first 15 lines):
# 📊 ResearchOS Daily Pipeline Summary
**Run ID:** `81d104d1-2541-4bea-8eb7-262e808fa381`
**Status:** ✅ Completed
**Duration:** 60m 26.3s

---

> 📅 **Pipeline Execution Metadata**
> - **Started:** 2026-02-04 06:54:59 UTC
> - **Ended:** 2026-02-04 07:55:25 UTC
> - **Total Duration:** 60m 26.3s
> - **Modules Executed:** 8

---

...


In [52]:
# ============================================================
# Cell 13 — Generate standard Markdown summary
# ============================================================
# Overview:
#   Generate a standard Markdown summary of the daily pipeline run.
#   Uses universal Markdown formatting (no Notion-specific features).
#   Includes run metadata, module execution status, aggregated metrics,
#   and error summaries. Suitable for Git README, wikis, or plain text viewers.
#
# Inputs / Outputs:
#   Inputs: orchestrator_state (with module_results, aggregated_metrics, errors)
#   Outputs: orchestrator_state['summaries']['standard_markdown'] (str)
#
# Notes:
#   - Universal Markdown: no callouts, standard tables only
#   - Compatible with GitHub, GitLab, basic Markdown parsers
#   - Emoji used sparingly for status (optional, widely supported)
#   - Stored in orchestrator_state for export in Cell 15
#   - Cleaner, more compact format than Notion version
#

print('[Cell 13] Generating standard Markdown summary...')
print('=' * 60)

# Reuse helper functions from Cell 12
# (format_duration, format_timestamp, status_emoji already defined)

# Extract orchestrator state
run_id = orchestrator_state.get('run_id', 'unknown')
pipeline_status = orchestrator_state.get('pipeline_status', 'unknown')
pipeline_start = orchestrator_state.get('pipeline_start')
pipeline_end = orchestrator_state.get('pipeline_end')
pipeline_duration = orchestrator_state.get('pipeline_duration_seconds', 0.0)
agg = orchestrator_state.get('aggregated_metrics', {})
module_results = orchestrator_state.get('module_results', {})
errors = orchestrator_state.get('errors', [])

# --- Build standard Markdown ---
standard_md = []

# Title and header
standard_md.append(f"# ResearchOS Daily Pipeline Summary\n\n")
standard_md.append(f"**Run ID:** `{run_id}`\n\n")
standard_md.append(f"**Status:** {status_emoji(pipeline_status)} {pipeline_status.replace('_', ' ').title()}\n\n")
standard_md.append(f"**Started:** {format_timestamp(pipeline_start)}\n\n")
standard_md.append(f"**Ended:** {format_timestamp(pipeline_end)}\n\n")
standard_md.append(f"**Duration:** {format_duration(pipeline_duration)}\n\n")
standard_md.append("---\n\n")

# Aggregated metrics
standard_md.append("## Aggregated Metrics\n\n")
standard_md.append("| Metric | Count |\n")
standard_md.append("|--------|-------|\n")
standard_md.append(f"| Papers Found | {agg.get('papers_found', 0)} |\n")
standard_md.append(f"| PDFs Processed | {agg.get('pdfs_processed', 0)} |\n")
standard_md.append(f"| VC Updates | {agg.get('vc_updates', 0)} |\n")
standard_md.append(f"| Startup Updates | {agg.get('startup_updates', 0)} |\n")
standard_md.append(f"| Policy Updates | {agg.get('policy_updates', 0)} |\n")
standard_md.append(f"| People Updates | {agg.get('people_updates', 0)} |\n")
standard_md.append(f"| **Total Events Created** | **{agg.get('events_created', 0)}** |\n")
standard_md.append("\n")

# Execution summary
standard_md.append("## Execution Summary\n\n")
standard_md.append(f"- **Modules Executed:** {agg.get('total_modules_executed', 0)}\n")
standard_md.append(f"- **Succeeded:** {agg.get('modules_succeeded', 0)} {status_emoji('success')}\n")
standard_md.append(f"- **Failed:** {agg.get('modules_failed', 0)} {status_emoji('failed')}\n")
standard_md.append(f"- **Skipped:** {agg.get('modules_skipped', 0)} {status_emoji('skipped')}\n")
standard_md.append(f"- **Total Execution Time:** {format_duration(agg.get('total_execution_time', 0.0))}\n")
standard_md.append("\n")

# Module execution details
standard_md.append("## Module Execution Details\n\n")
standard_md.append("| Module | Status | Duration | Key Metric |\n")
standard_md.append("|--------|--------|----------|------------|\n")

for module_name, result in module_results.items():
    status = result.get('status', 'unknown')
    emoji = status_emoji(status)
    duration = format_duration(result.get('duration_seconds', 0.0))
    
    # Extract key metric
    key_metric = '—'
    if result.get('metrics'):
        metrics = result['metrics']
        if 'papers_found' in metrics:
            key_metric = f"{metrics['papers_found']} papers"
        elif 'pdfs_processed' in metrics:
            key_metric = f"{metrics['pdfs_processed']} PDFs"
        elif 'updates_found' in metrics:
            key_metric = f"{metrics['updates_found']} updates"
        elif 'events_created' in metrics:
            key_metric = f"{metrics['events_created']} events"
    
    standard_md.append(f"| {module_name} | {emoji} {status} | {duration} | {key_metric} |\n")

standard_md.append("\n")

# Errors (if any)
if errors:
    standard_md.append("## Errors\n\n")
    for i, error in enumerate(errors, 1):
        critical_flag = ' **[CRITICAL]**' if error.get('critical', False) else ''
        standard_md.append(f"{i}. **{error.get('module', 'Unknown')}{critical_flag}**\n")
        standard_md.append(f"   - Error: `{error.get('error', 'No details')}`\n")
        standard_md.append(f"   - Time: {format_timestamp(error.get('timestamp'))}\n")
        standard_md.append("\n")
else:
    standard_md.append("## Status\n\n")
    standard_md.append(f"{status_emoji('success')} No errors encountered.\n\n")

# Footer
standard_md.append("---\n\n")
standard_md.append(f"*Generated: {format_timestamp(pipeline_end)}*\n")

# Join and store
standard_markdown = ''.join(standard_md)
orchestrator_state['summaries']['standard_markdown'] = standard_markdown

print(f'[SUCCESS] Standard Markdown summary generated')
print(f'  - Length: {len(standard_markdown)} characters')
print(f'  - Sections: Metadata, metrics, execution details, errors')
print(f'  - Format: Universal Markdown (GitHub/GitLab compatible)')
print('=' * 60)

# Preview first few lines
preview_lines = standard_markdown.split('\n')[:12]
print('\nPreview (first 12 lines):')
print('\n'.join(preview_lines))
print('...')


[Cell 13] Generating standard Markdown summary...
[SUCCESS] Standard Markdown summary generated
  - Length: 1200 characters
  - Sections: Metadata, metrics, execution details, errors
  - Format: Universal Markdown (GitHub/GitLab compatible)

Preview (first 12 lines):
# ResearchOS Daily Pipeline Summary

**Run ID:** `81d104d1-2541-4bea-8eb7-262e808fa381`

**Status:** ✅ Completed

**Started:** 2026-02-04 06:54:59 UTC

**Ended:** 2026-02-04 07:55:25 UTC

**Duration:** 60m 26.3s

...


In [53]:
# ============================================================
# Cell 14 — Generate Slack text snippet summary
# ============================================================
# Overview:
#   Generate a concise Slack-formatted text summary of the daily pipeline run.
#   Optimized for Slack's text formatting (bold, code blocks, bullets).
#   Includes run metadata, high-level metrics, status summary, and error alerts.
#   Designed for posting as a Slack message or file snippet.
#
# Inputs / Outputs:
#   Inputs: orchestrator_state (with module_results, aggregated_metrics, errors)
#   Outputs: orchestrator_state['summaries']['slack_text'] (str)
#
# Notes:
#   - Slack formatting: *bold*, `code`, ```code blocks```, • bullets
#   - Compact format suitable for channel posting or snippet upload
#   - No tables (limited Slack support), uses structured bullet lists
#   - Emoji used for visual status indicators
#   - Stored in orchestrator_state for export in Cell 15
#   - Target length: ~20-40 lines for readability in Slack
#

print('[Cell 14] Generating Slack text snippet summary...')
print('=' * 60)

# Reuse helper functions from Cell 12
# (format_duration, format_timestamp, status_emoji already defined)

# Extract orchestrator state
run_id = orchestrator_state.get('run_id', 'unknown')
pipeline_status = orchestrator_state.get('pipeline_status', 'unknown')
pipeline_start = orchestrator_state.get('pipeline_start')
pipeline_end = orchestrator_state.get('pipeline_end')
pipeline_duration = orchestrator_state.get('pipeline_duration_seconds', 0.0)
agg = orchestrator_state.get('aggregated_metrics', {})
module_results = orchestrator_state.get('module_results', {})
errors = orchestrator_state.get('errors', [])

# --- Build Slack text snippet ---
slack_lines = []

# Header
slack_lines.append(f"*ResearchOS Daily Pipeline Summary*")
slack_lines.append(f"")
slack_lines.append(f"*Run ID:* `{run_id}`")
slack_lines.append(f"*Status:* {status_emoji(pipeline_status)} *{pipeline_status.replace('_', ' ').title()}*")
slack_lines.append(f"*Duration:* {format_duration(pipeline_duration)}")
slack_lines.append(f"*Completed:* {format_timestamp(pipeline_end)}")
slack_lines.append(f"")
slack_lines.append(f"---")
slack_lines.append(f"")

# Metrics summary
slack_lines.append(f"*📊 Metrics Summary*")
slack_lines.append(f"• Papers Found: *{agg.get('papers_found', 0)}*")
slack_lines.append(f"• PDFs Processed: *{agg.get('pdfs_processed', 0)}*")
slack_lines.append(f"• VC Updates: *{agg.get('vc_updates', 0)}*")
slack_lines.append(f"• Startup Updates: *{agg.get('startup_updates', 0)}*")
slack_lines.append(f"• Policy Updates: *{agg.get('policy_updates', 0)}*")
slack_lines.append(f"• People Updates: *{agg.get('people_updates', 0)}*")
slack_lines.append(f"• *Total Events Created: {agg.get('events_created', 0)}*")
slack_lines.append(f"")

# Module execution summary
slack_lines.append(f"*🔧 Module Execution*")
slack_lines.append(f"• Total Modules: {agg.get('total_modules_executed', 0)}")
slack_lines.append(f"• {status_emoji('success')} Succeeded: {agg.get('modules_succeeded', 0)}")
if agg.get('modules_failed', 0) > 0:
    slack_lines.append(f"• {status_emoji('failed')} Failed: {agg.get('modules_failed', 0)}")
if agg.get('modules_skipped', 0) > 0:
    slack_lines.append(f"• {status_emoji('skipped')} Skipped: {agg.get('modules_skipped', 0)}")
slack_lines.append(f"• Execution Time: {format_duration(agg.get('total_execution_time', 0.0))}")
slack_lines.append(f"")

# Per-module status (compact)
slack_lines.append(f"*Module Status:*")
for module_name, result in module_results.items():
    status = result.get('status', 'unknown')
    emoji = status_emoji(status)
    duration = format_duration(result.get('duration_seconds', 0.0))
    
    # Extract key metric if available
    metric_str = ''
    if result.get('metrics'):
        metrics = result['metrics']
        if 'papers_found' in metrics:
            metric_str = f" ({metrics['papers_found']} papers)"
        elif 'pdfs_processed' in metrics:
            metric_str = f" ({metrics['pdfs_processed']} PDFs)"
        elif 'updates_found' in metrics:
            metric_str = f" ({metrics['updates_found']} updates)"
        elif 'events_created' in metrics:
            metric_str = f" ({metrics['events_created']} events)"
    
    slack_lines.append(f"• {emoji} `{module_name}` - {duration}{metric_str}")

slack_lines.append(f"")

# Errors (if any)
if errors:
    slack_lines.append(f"*⚠️ Errors Encountered ({len(errors)})*")
    for error in errors[:3]:  # Limit to first 3 for brevity
        critical_flag = ' [CRITICAL]' if error.get('critical', False) else ''
        slack_lines.append(f"• `{error.get('module', 'Unknown')}{critical_flag}`: {error.get('error', 'No details')[:80]}...")
    
    if len(errors) > 3:
        slack_lines.append(f"• _(+{len(errors) - 3} more errors - see full log)_")
    
    slack_lines.append(f"")
else:
    slack_lines.append(f"*✅ No Errors*")
    slack_lines.append(f"")

# Footer
slack_lines.append(f"---")
slack_lines.append(f"Pipeline: {status_emoji(pipeline_status)} *{pipeline_status.replace('_', ' ').title()}*")

# Join and store
slack_text = '\n'.join(slack_lines)
orchestrator_state['summaries']['slack_text'] = slack_text

print(f'[SUCCESS] Slack text snippet summary generated')
print(f'  - Length: {len(slack_text)} characters')
print(f'  - Lines: {len(slack_lines)}')
print(f'  - Sections: Header, metrics, module status, errors, footer')
print(f'  - Format: Slack-compatible text with emoji and formatting')
print('=' * 60)

# Preview
preview_lines = slack_text.split('\n')[:20]
print('\nPreview (first 20 lines):')
print('\n'.join(preview_lines))
if len(slack_lines) > 20:
    print('...')
print('\n' + '=' * 60)


[Cell 14] Generating Slack text snippet summary...
[SUCCESS] Slack text snippet summary generated
  - Length: 804 characters
  - Lines: 37
  - Sections: Header, metrics, module status, errors, footer
  - Format: Slack-compatible text with emoji and formatting

Preview (first 20 lines):
*ResearchOS Daily Pipeline Summary*

*Run ID:* `81d104d1-2541-4bea-8eb7-262e808fa381`
*Status:* ✅ *Completed*
*Duration:* 60m 26.3s
*Completed:* 2026-02-04 07:55:25 UTC

---

*📊 Metrics Summary*
• Papers Found: *0*
• PDFs Processed: *1*
• VC Updates: *0*
• Startup Updates: *0*
• Policy Updates: *0*
• People Updates: *1*
• *Total Events Created: 2*

*🔧 Module Execution*
• Total Modules: 8
...



In [54]:
# ============================================================
# Cell 15 — Display final summary and save artifacts
# ============================================================
# Overview:
#   Display the final pipeline summary to the notebook output.
#   Save all three summary formats (Notion Markdown, standard Markdown,
#   Slack text) to files for downstream consumption.
#   Export orchestrator_state as JSON for audit/debugging.
#   Print final status, metrics, and artifact locations.
#
# Inputs / Outputs:
#   Inputs: orchestrator_state (with summaries, module_results, aggregated_metrics)
#   Outputs: Printed summary to notebook output,
#            Files: daily_summary_notion.md, daily_summary.md, daily_summary_slack.txt,
#                   orchestrator_state.json
#
# Notes:
#   - All files saved to current working directory (same as notebooks)
#   - Filenames include run_id for traceability (if available)
#   - JSON export includes full orchestrator_state for audit trail
#   - Defensively handles missing summaries or file write errors
#   - Final cell marks pipeline completion
#

print('[Cell 15] Displaying final summary and saving artifacts...')
print('=' * 60)

import json
import os
from pathlib import Path

# --- Display Pipeline Summary to Notebook Output ---
print('\n' + '=' * 60)
print('RESEARCHOS DAILY PIPELINE - FINAL SUMMARY')
print('=' * 60)

# Pipeline metadata
run_id = orchestrator_state.get('run_id', 'unknown')
pipeline_status = orchestrator_state.get('pipeline_status', 'unknown')
pipeline_duration = orchestrator_state.get('pipeline_duration_seconds', 0.0)
agg = orchestrator_state.get('aggregated_metrics', {})

print(f"\nRun ID: {run_id}")
print(f"Status: {pipeline_status.replace('_', ' ').title()}")
print(f"Duration: {format_duration(pipeline_duration)}")
print(f"Completed: {format_timestamp(orchestrator_state.get('pipeline_end'))}")

# High-level metrics
print(f"\nMetrics Summary:")
print(f"  - Papers Found: {agg.get('papers_found', 0)}")
print(f"  - PDFs Processed: {agg.get('pdfs_processed', 0)}")
print(f"  - VC Updates: {agg.get('vc_updates', 0)}")
print(f"  - Startup Updates: {agg.get('startup_updates', 0)}")
print(f"  - Policy Updates: {agg.get('policy_updates', 0)}")
print(f"  - People Updates: {agg.get('people_updates', 0)}")
print(f"  - Total Events Created: {agg.get('events_created', 0)}")

# Execution summary
print(f"\nExecution Summary:")
print(f"  - Modules Executed: {agg.get('total_modules_executed', 0)}")
print(f"  - Succeeded: {agg.get('modules_succeeded', 0)}")
print(f"  - Failed: {agg.get('modules_failed', 0)}")
print(f"  - Skipped: {agg.get('modules_skipped', 0)}")
print(f"  - Total Execution Time: {format_duration(agg.get('total_execution_time', 0.0))}")

# Error summary
errors = orchestrator_state.get('errors', [])
if errors:
    print(f"\n⚠️  Errors Encountered: {len(errors)}")
    for i, error in enumerate(errors[:5], 1):  # Show first 5
        critical = ' [CRITICAL]' if error.get('critical', False) else ''
        print(f"  {i}. {error.get('module', 'Unknown')}{critical}")
        print(f"     {error.get('error', 'No details')[:100]}")
    if len(errors) > 5:
        print(f"  ... and {len(errors) - 5} more (see full logs)")
else:
    print(f"\n✅ No errors encountered")

print('\n' + '=' * 60)

# --- Save Artifacts to Files ---
print('\nSaving summary artifacts...')

# Prepare filenames with run_id if available
if run_id and run_id != 'unknown':
    notion_filename = f'daily_summary_notion_{run_id}.md'
    standard_filename = f'daily_summary_{run_id}.md'
    slack_filename = f'daily_summary_slack_{run_id}.txt'
    state_filename = f'orchestrator_state_{run_id}.json'
else:
    notion_filename = 'daily_summary_notion.md'
    standard_filename = 'daily_summary.md'
    slack_filename = 'daily_summary_slack.txt'
    state_filename = 'orchestrator_state.json'

artifacts_saved = []
artifacts_failed = []

# Save Notion Markdown
try:
    notion_md = orchestrator_state['summaries'].get('notion_markdown')
    if notion_md:
        with open(notion_filename, 'w', encoding='utf-8') as f:
            f.write(notion_md)
        artifacts_saved.append(notion_filename)
        print(f"  ✓ Saved Notion Markdown: {notion_filename} ({len(notion_md)} chars)")
    else:
        artifacts_failed.append((notion_filename, 'No content generated'))
        print(f"  ✗ Notion Markdown not generated")
except Exception as e:
    artifacts_failed.append((notion_filename, str(e)))
    print(f"  ✗ Failed to save {notion_filename}: {e}")

# Save standard Markdown
try:
    standard_md = orchestrator_state['summaries'].get('standard_markdown')
    if standard_md:
        with open(standard_filename, 'w', encoding='utf-8') as f:
            f.write(standard_md)
        artifacts_saved.append(standard_filename)
        print(f"  ✓ Saved standard Markdown: {standard_filename} ({len(standard_md)} chars)")
    else:
        artifacts_failed.append((standard_filename, 'No content generated'))
        print(f"  ✗ Standard Markdown not generated")
except Exception as e:
    artifacts_failed.append((standard_filename, str(e)))
    print(f"  ✗ Failed to save {standard_filename}: {e}")

# Save Slack text snippet
try:
    slack_text = orchestrator_state['summaries'].get('slack_text')
    if slack_text:
        with open(slack_filename, 'w', encoding='utf-8') as f:
            f.write(slack_text)
        artifacts_saved.append(slack_filename)
        print(f"  ✓ Saved Slack text snippet: {slack_filename} ({len(slack_text)} chars)")
    else:
        artifacts_failed.append((slack_filename, 'No content generated'))
        print(f"  ✗ Slack text snippet not generated")
except Exception as e:
    artifacts_failed.append((slack_filename, str(e)))
    print(f"  ✗ Failed to save {slack_filename}: {e}")

# Save orchestrator state as JSON (for audit/debugging)
try:
    # Prepare state for JSON serialization (convert datetime objects if needed)
    state_export = orchestrator_state.copy()
    
    with open(state_filename, 'w', encoding='utf-8') as f:
        json.dump(state_export, f, indent=2, default=str)
    
    artifacts_saved.append(state_filename)
    file_size = os.path.getsize(state_filename)
    print(f"  ✓ Saved orchestrator state: {state_filename} ({file_size} bytes)")
except Exception as e:
    artifacts_failed.append((state_filename, str(e)))
    print(f"  ✗ Failed to save {state_filename}: {e}")

print(f"\nArtifacts saved: {len(artifacts_saved)}")
if artifacts_failed:
    print(f"Artifacts failed: {len(artifacts_failed)}")
    for filename, reason in artifacts_failed:
        print(f"  - {filename}: {reason}")

# --- Export Summary Variables for Notebook Access ---
# Make summary strings available as top-level variables for downstream use
daily_summary_md_notion = orchestrator_state['summaries'].get('notion_markdown', '')
daily_summary_md = orchestrator_state['summaries'].get('standard_markdown', '')
daily_summary_slack = orchestrator_state['summaries'].get('slack_text', '')

print(f"\nExported variables for notebook access:")
print(f"  - daily_summary_md_notion: {len(daily_summary_md_notion)} chars")
print(f"  - daily_summary_md: {len(daily_summary_md)} chars")
print(f"  - daily_summary_slack: {len(daily_summary_slack)} chars")
print(f"  - orchestrator_state: {len(orchestrator_state)} keys")

# --- Final Status ---
print('\n' + '=' * 60)
print('PIPELINE COMPLETE')
print('=' * 60)
print(f"\nStatus: {pipeline_status.replace('_', ' ').title()}")
print(f"Run ID: {run_id}")
print(f"Total Events Created: {agg.get('events_created', 0)}")
print(f"\nArtifacts:")
for artifact in artifacts_saved:
    print(f"  - {artifact}")

if pipeline_status == 'completed':
    print(f"\n✅ All modules completed successfully.")
elif pipeline_status == 'completed_with_failures':
    print(f"\n⚠️  Pipeline completed with {agg.get('modules_failed', 0)} module failure(s).")
    print(f"   Review error logs above for details.")
else:
    print(f"\n❌ Pipeline failed. See error logs above.")

print(f"\nNext steps:")
print(f"  1. Review Event entries in Notion")
print(f"  2. Check summary files: {', '.join(artifacts_saved[:3])}")
if artifacts_failed:
    print(f"  3. Investigate failed artifacts: {len(artifacts_failed)} issue(s)")
if errors:
    print(f"  4. Debug failed modules: {agg.get('modules_failed', 0)} error(s)")

print('\n' + '=' * 60)
print('[Cell 15] Orchestrator execution complete.')
print('=' * 60)


[Cell 15] Displaying final summary and saving artifacts...

RESEARCHOS DAILY PIPELINE - FINAL SUMMARY

Run ID: 81d104d1-2541-4bea-8eb7-262e808fa381
Status: Completed
Duration: 60m 26.3s
Completed: 2026-02-04 07:55:25 UTC

Metrics Summary:
  - Papers Found: 0
  - PDFs Processed: 1
  - VC Updates: 0
  - Startup Updates: 0
  - Policy Updates: 0
  - People Updates: 1
  - Total Events Created: 2

Execution Summary:
  - Modules Executed: 8
  - Succeeded: 8
  - Failed: 0
  - Skipped: 0
  - Total Execution Time: 7m 16.3s

✅ No errors encountered


Saving summary artifacts...
  ✓ Saved Notion Markdown: daily_summary_notion_81d104d1-2541-4bea-8eb7-262e808fa381.md (3469 chars)
  ✓ Saved standard Markdown: daily_summary_81d104d1-2541-4bea-8eb7-262e808fa381.md (1200 chars)
  ✓ Saved Slack text snippet: daily_summary_slack_81d104d1-2541-4bea-8eb7-262e808fa381.txt (804 chars)
  ✓ Saved orchestrator state: orchestrator_state_81d104d1-2541-4bea-8eb7-262e808fa381.json (13861 bytes)

Artifacts saved: 4

